# Stage 01 — Review (T1 + T3)

For every patent in `raw_images/`:
1. OCR each image → figure label (from the image itself)
2. Read BRIEF DESCRIPTION from `text/<patent_id>.txt` (written by Stage 00 from EPO)
3. Match OCR label → description line → assign `match_status`
4. Auto-fill visual taxonomy fields from description keywords
5. Write `labels/<patent_id>.json`
6. Render review table: thumbnail | OCR label | description | visual fields | status

**match_status colour key:**
- 🟢 `matched` — OCR label found and matched to a description line (confidence 0.95)
- 🔴 `unmatched` — OCR label found but no matching line in description text (confidence 0.0)
- 🟡 `no_label` — OCR found nothing; positional fallback used (confidence 0.3)
- 🟠 `duplicate` — same figure number claimed by multiple images (confidence 0.9)

Rows with `needs_review: true` are **highlighted with a yellow border**.

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from src.config_loader import load_config
from src.extractor import load_patseer_excel
from src.reviewer import process_patent

cfg = load_config()
raw_dir    = cfg['paths']['raw_images']
text_dir   = cfg['paths']['text']
labels_dir = cfg['paths']['labels']
print(f"raw_images : {raw_dir}")
print(f"text       : {text_dir}")
print(f"labels     : {labels_dir}")

In [ ]:
# Excel used for T1 metadata only (title, assignee, years, citations).
# Drawing descriptions are read from text/<patent_id>.txt by process_patent().
excel_index = load_patseer_excel(cfg['paths']['patseer_excel'])
print(f"Indexed {len(excel_index)} patents.")

In [ ]:
# Run OCR + matching + JSON assembly for all downloaded patents
patent_dirs = sorted(raw_dir.iterdir()) if raw_dir.exists() else []
json_paths = {}

for patent_dir in patent_dirs:
    if not patent_dir.is_dir():
        continue
    patent_id = patent_dir.name

    # Check if description text file exists
    txt_exists = (text_dir / f"{patent_id}.txt").exists()
    print(f'Processing {patent_id}…  (desc: {"✓" if txt_exists else "✗ missing"})', end=' ')

    json_path = process_patent(patent_id, cfg, excel_index, raw_dir)
    json_paths[patent_id] = json_path
    print(f'→ {json_path.name}')

print(f'\nWrote {len(json_paths)} JSON files.')

In [ ]:
# ── Build review table ────────────────────────────────────────────────────────
import json
import base64
import io
import pandas as pd
from PIL import Image
from IPython.display import display, HTML

THUMB_PX = 120

def _thumb_html(img_path: Path) -> str:
    try:
        img = Image.open(img_path)
        img.thumbnail((THUMB_PX, THUMB_PX))
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        b64 = base64.b64encode(buf.getvalue()).decode()
        return f'<img src="data:image/png;base64,{b64}" style="max-height:{THUMB_PX}px"/>'
    except Exception:
        return '(no image)'

def _visual_summary(visual: dict) -> str:
    parts = []
    for field in ['perspective', 'style', 'symmetry']:
        v = visual.get(field, {})
        if v.get('value'):
            src = '🤖' if v.get('source') == 'auto' else '✋'
            parts.append(f"{src} {v['value']}")
    return '<br>'.join(parts) if parts else '—'

rows = []
for patent_id, json_path in json_paths.items():
    data = json.loads(json_path.read_text(encoding='utf-8'))
    for img in data.get('T3_images', []):
        img_path = raw_dir / patent_id / img['file']
        rows.append({
            'patent_id':    patent_id,
            'thumbnail':    _thumb_html(img_path) if img_path.exists() else '(missing)',
            'file':         img['file'],
            'ocr_label':    img['ocr_label'] or '—',
            'match_status': img['match_status'],
            'confidence':   img['match_confidence'],
            'description':  (img['matched_description'] or '—')[:120],
            'visual':       _visual_summary(img.get('visual', {})),
            'needs_review': img['needs_review'],
        })

df = pd.DataFrame(rows)
print(f"Total images in review: {len(df)}")
print(f"Needs review           : {df['needs_review'].sum()}")

In [ ]:
# ── Render styled HTML table ──────────────────────────────────────────────────
STATUS_COLORS = {
    'matched':   '#d4edda',
    'unmatched': '#f8d7da',
    'no_label':  '#fff3cd',
    'duplicate': '#fde2e4',
}

def _row_style(row):
    bg = STATUS_COLORS.get(row['match_status'], '')
    border = 'border: 2px solid #e0a800;' if row['needs_review'] else ''
    return [f'background-color: {bg}; {border}'] * len(row)

styled = (
    df.style
    .apply(_row_style, axis=1)
    .format({
        'thumbnail':   lambda x: x,
        'visual':      lambda x: x,
        'confidence':  '{:.2f}',
    })
    .set_table_styles([{'selector': 'td', 'props': [('vertical-align', 'middle')]}])
    .hide(axis='index')
)

display(HTML(styled.to_html(escape=False)))